In [1]:
import argparse
import glob
import os
from datetime import datetime
import numpy as np
import pandas as pd

# 1. CONFIGURATION
pd.set_option('display.float_format', lambda x: '%.2f' % x)

BASE_DIR = '..'
CITY_FOLDER = 'brussels'  
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed', CITY_FOLDER)
KPI_OUTPUT_DIR = os.path.join(BASE_DIR, 'processed', CITY_FOLDER) 

print(f"--- STARTING FULL KPI CALCULATION FOR: {CITY_FOLDER.upper()} ---")

# 2. FUNCTION TO CALCULATE KPI FOR EACH SNAPSHOT
def calculate_snapshot_kpi(snapshot_path):
    snapshot_name = os.path.basename(snapshot_path)
    print(f"\n>> Processing snapshot: {snapshot_name}")
    
    # --- Load Data ---
    try:
        df_list = pd.read_csv(os.path.join(snapshot_path, 'listings_processed.csv'), low_memory=False)
        df_cal = pd.read_csv(os.path.join(snapshot_path, 'calendar_processed.csv'), low_memory=False)
        
        try:
            df_rev = pd.read_csv(os.path.join(snapshot_path, 'reviews_processed.csv'), low_memory=False)
            df_rev['date'] = pd.to_datetime(df_rev['date'])
        except FileNotFoundError:
            df_rev = pd.DataFrame(columns=['listing_id', 'date']) # Create empty dataframe if missing
            
    except FileNotFoundError:
        print("   [!] Listings or calendar file not found. Skipping.")
        return None, None, None, None, None

    # --- QA Filtering (Data Cleansing) ---
    # Only keep listings that are not flagged for zero price or out of bounds coordinates
    # (If the QA flag columns do not exist, assume the records are valid)
    if 'qa_flag_price_zero' in df_list.columns:
        df_list = df_list[df_list['qa_flag_price_zero'] == False]
    if 'qa_flag_out_of_city' in df_list.columns:
        df_list = df_list[df_list['qa_flag_out_of_city'] == False]
        
    valid_listings = df_list.copy()
    valid_ids = valid_listings['id'].unique()
    
    # Filter calendar entries to match valid listings only
    valid_cal = df_cal[df_cal['listing_id'].isin(valid_ids)].copy()

    # ==============================================================================
    # PART 1: OVERALL SUMMARY & ROOM TYPE
    # ==============================================================================
    total_supply = len(valid_listings)
    
    # Multi-hosts rate
    host_counts = valid_listings['host_id'].value_counts()
    multi_host_rate = (host_counts[host_counts > 1].sum() / total_supply) * 100 if total_supply > 0 else 0

    # Median Price & Availability
    median_price = valid_listings['price_numeric'].median()
    median_avail_90 = valid_listings['availability_90'].median()
    
    # Room Type Stats
    room_stats = valid_listings['room_type'].value_counts(normalize=True).reset_index()
    room_stats.columns = ['room_type', 'percentage']
    room_stats['percentage'] = room_stats['percentage'] * 100
    room_stats['snapshot_date'] = snapshot_name

    # Summary Row
    summary_row = {
        'snapshot_date': snapshot_name,
        'total_listings': total_supply,
        'multi_host_rate': multi_host_rate,
        'median_price': median_price,
        'median_avail_90': median_avail_90
    }

    # ==============================================================================
    # PART 2: SEASONALITY (CALENDAR)
    # ==============================================================================
    valid_cal['date'] = pd.to_datetime(valid_cal['date'])
    valid_cal['is_booked'] = valid_cal['available'].apply(lambda x: 1 if x == 'f' else 0)
    
    daily_occupancy = valid_cal.groupby('date')['is_booked'].mean().reset_index()
    daily_occupancy.columns = ['date', 'occupancy_rate']
    daily_occupancy['snapshot_date'] = snapshot_name

    # ==============================================================================
    # PART 3: NEIGHBOURHOOD 
    # ==============================================================================
    # Use either neighbourhood_cleansed or neighbourhood column based on availability
    col_neigh = 'neighbourhood_cleansed' if 'neighbourhood_cleansed' in valid_listings.columns else 'neighbourhood'
    
    neigh_stats = valid_listings.groupby(col_neigh).agg({
        'id': 'count',
        'price_numeric': 'median'
    }).reset_index()
    neigh_stats.columns = ['neighbourhood', 'listing_count', 'median_price']
    neigh_stats['snapshot_date'] = snapshot_name

    # ==============================================================================
    # PART 4: REVIEW TRENDS 
    # ==============================================================================
    # Resample by month
    if not df_rev.empty:
        review_trends = df_rev.set_index('date').resample('M').size().reset_index(name='review_count')
        review_trends['snapshot_date'] = snapshot_name
        # Optional: You can filter for the last 2 years relative to the snapshot here to reduce noise
    else:
        review_trends = pd.DataFrame()

    return summary_row, room_stats, daily_occupancy, neigh_stats, review_trends

# 3. RUN PIPELINE & AGGREGATE RESULTS
all_snapshots = glob.glob(os.path.join(PROCESSED_DIR, '*'))
summary_list = []
room_stats_list = []
occupancy_list = []
neigh_stats_list = []
review_stats_list = []

for folder in all_snapshots:
    if os.path.isdir(folder):
        res_sum, res_room, res_occ, res_neigh, res_rev = calculate_snapshot_kpi(folder)
        if res_sum:
            summary_list.append(res_sum)
            room_stats_list.append(res_room)
            occupancy_list.append(res_occ)
            neigh_stats_list.append(res_neigh)
            if not res_rev.empty: review_stats_list.append(res_rev)

# 4. CONSOLIDATE & EXPORT FILES
if summary_list:
    # 4.1 General Summary
    df_summary = pd.DataFrame(summary_list).sort_values('snapshot_date')
    df_summary.to_csv(os.path.join(KPI_OUTPUT_DIR, f'kpi_summary_general_{CITY_FOLDER}.csv'), index=False)
    
    # 4.2 Room Type Stats
    if room_stats_list:
        pd.concat(room_stats_list).to_csv(os.path.join(KPI_OUTPUT_DIR, f'kpi_room_type_{CITY_FOLDER}.csv'), index=False)
    
    # 4.3 Seasonality Stats
    if occupancy_list:
        pd.concat(occupancy_list).to_csv(os.path.join(KPI_OUTPUT_DIR, f'kpi_seasonality_{CITY_FOLDER}.csv'), index=False)

    # 4.4 Neighbourhood Stats
    if neigh_stats_list:
        pd.concat(neigh_stats_list).to_csv(os.path.join(KPI_OUTPUT_DIR, f'kpi_neighbourhood_{CITY_FOLDER}.csv'), index=False)

    # 4.5 Review Trends Stats
    if review_stats_list:
        # Concatenate all generated trend data across snapshots
        pd.concat(review_stats_list).to_csv(os.path.join(KPI_OUTPUT_DIR, f'kpi_reviews_trend_{CITY_FOLDER}.csv'), index=False)

    print("\n COMPLETED! Created 5 KPI files:")
    print(f"   1. kpi_summary_general_{CITY_FOLDER}.csv")
    print(f"   2. kpi_room_type_{CITY_FOLDER}.csv")
    print(f"   3. kpi_seasonality_{CITY_FOLDER}.csv")
    print(f"   4. kpi_neighbourhood_{CITY_FOLDER}.csv")
    print(f"   5. kpi_reviews_trend_{CITY_FOLDER}.csv")

else:
    print("No data available to calculate metrics.")

--- STARTING FULL KPI CALCULATION FOR: BRUSSELS ---

>> Processing snapshot: 16 March, 2025


C:\Users\Admin\AppData\Local\Temp\ipykernel_15524\3881047139.py:108: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  review_trends = df_rev.set_index('date').resample('M').size().reset_index(name='review_count')



>> Processing snapshot: 21 June, 2025


C:\Users\Admin\AppData\Local\Temp\ipykernel_15524\3881047139.py:108: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  review_trends = df_rev.set_index('date').resample('M').size().reset_index(name='review_count')



>> Processing snapshot: 22 December, 2024

 COMPLETED! Created 5 KPI files:
   1. kpi_summary_general_brussels.csv
   2. kpi_room_type_brussels.csv
   3. kpi_seasonality_brussels.csv
   4. kpi_neighbourhood_brussels.csv
   5. kpi_reviews_trend_brussels.csv


C:\Users\Admin\AppData\Local\Temp\ipykernel_15524\3881047139.py:108: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  review_trends = df_rev.set_index('date').resample('M').size().reset_index(name='review_count')
